In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp
import warnings
from scipy.optimize import minimize
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

output_file         = '../05_output/'
osap_data           = '../03_data/osap/'
crsp                = 'F:/Project/00_wrds data/crsp/'
firm_signal         = 'F:/Project/00_wrds data/firm_characteristics/signed_predictors_dl_wide/'
crsp_signal         = 'F:/Project/00_wrds data/firm_characteristics/signals/'
signal_doc          = 'F:/Project/00_wrds data/firm_characteristics/signal_list/'

d:\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def fast_pivot_3d(df, index_col, columns_col, value_cols):
    row_idx = pd.Categorical(df[index_col], ordered=True)
    col_idx = pd.Categorical(df[columns_col], ordered=True)
    
    n_rows = len(row_idx.categories)
    n_cols = len(col_idx.categories)
    n_feats = len(value_cols)

    out = np.full((n_rows, n_cols, n_feats), np.nan)
    out[row_idx.codes, col_idx.codes, :] = df[value_cols].values
    
    return out, row_idx.categories, col_idx.categories

In [3]:
##| stock data
stock_monthly = pd.DataFrame([])
for year in tqdm(range(1960, 2026)):
    mse             = pd.read_pickle(crsp + f'mseall/mseall{year}.pickle')
    msf             = pd.read_pickle(crsp + f'msf/msf{year}.pickle')
    msf             = msf.merge(right=mse[['permno', 'date', 'exchcd', 'shrcd']], on=['permno', 'date'], how='outer')

    stock_monthly = pd.concat(
        [stock_monthly, msf[['permno', 'date', 'ret', 'shrout', 'shrcd', 'exchcd']]],
        axis=0,
    )

stock_monthly[['exchcd', 'shrcd']] = stock_monthly.groupby(['permno'])[['exchcd', 'shrcd']].ffill()
stock_monthly = stock_monthly[
    (stock_monthly['ret'] >= -1)  
    &(stock_monthly['shrout'] == stock_monthly['shrout'])
    &(stock_monthly['shrcd'].isin([10, 11]))
    &(stock_monthly['exchcd'] == 1)
]
ret, stock1, date1 = fast_pivot_3d(stock_monthly, 'permno', 'date', ['ret'])


100%|██████████| 66/66 [00:18<00:00,  3.59it/s]


In [4]:
##| signal data
signal_list         = pd.read_csv(signal_doc + 'SignalDoc.csv')
signals             = pd.read_pickle(firm_signal + 'firm_signals.pickle')
Price               = pd.read_pickle(crsp_signal + 'Price.pickle')
Size                = pd.read_pickle(crsp_signal + 'Size.pickle')
STreversal          = pd.read_pickle(crsp_signal + 'STreversal.pickle')

#|  keep continuous characteristics
signal_list         = signal_list[
                        (signal_list['Cat.Form'] == 'continuous') 
                        & (signal_list['Cat.Data'].isin(['Price', 'Trading', 'Accounting', 'Analyst']))
                    ]['Acronym'].tolist()

keep_col            = [c for c in signals.columns if c in ['permno', 'yyyymm'] + signal_list]
signals             = signals[keep_col]

signals             = signals[signals['permno'].isin(list(stock1)) & (signals['yyyymm'] >= 197012)]
Price               = Price[Price['permno'].isin(list(stock1)) & (Price['yyyymm'] >= 197012)]
Size                = Size[Size['permno'].isin(list(stock1)) & (Size['yyyymm'] >= 197012)]
STreversal          = STreversal[STreversal['permno'].isin(list(stock1)) & (STreversal['yyyymm'] >= 197012)]

signals             = signals.merge(right=Price,      on=['permno', 'yyyymm'], how='outer')
signals             = signals.merge(right=Size,       on=['permno', 'yyyymm'], how='outer')
signals             = signals.merge(right=STreversal, on=['permno', 'yyyymm'], how='outer')

#|  generate array data
signalnames         = signals.columns[2:]
signals, stock2, date2 = fast_pivot_3d(signals, 'permno', 'yyyymm', list(signalnames))

In [5]:
##| save data
#|  get common stock and common date
date1               = np.array([int(d.split('-')[0]+d.split('-')[1]) for d in date1])
date2               = np.array(date2)
stock1              = np.array(stock1)
stock2              = np.array(stock2)

assert np.all(np.diff(stock1) > 0) and np.all(np.diff(stock2) > 0)
assert np.all(np.diff(date1)  > 0) and np.all(np.diff(date2)  > 0) # check point

#|  lag signal data for 1 period
signals             = signals[:, :-1, :]
date2               = date2  [1:]

common_stocks       = np.intersect1d(stock1, stock2)
common_dates        = np.intersect1d(date1, date2)

idx_stock1          = np.searchsorted(stock1, common_stocks)
idx_stock2          = np.searchsorted(stock2, common_stocks)
idx_date1           = np.searchsorted(date1, common_dates)
idx_date2           = np.searchsorted(date2, common_dates)

ret                 = ret    [np.ix_(idx_stock1, idx_date1)]
signals             = signals[np.ix_(idx_stock2, idx_date2)]

In [6]:
#|  save date
np.save(osap_data+'signalnames.npy', signalnames)
np.save(osap_data+'ret.npy'         , ret)
np.save(osap_data+'stock.npy'       , common_stocks)
np.save(osap_data+'date.npy'        , common_dates)
np.save(osap_data+'signals.npy'     , signals)

In [13]:
##| data summary
#|  number of signals
print(f'number of signals: {signals.shape[-1]}')

#|  number of firms
print(f'number of firms:   {signals.shape[0]}')

#|  number of firms
counts = np.sum(~np.isnan(ret), axis=0)
print(f'max firm number:   {counts.max()}')
print(f'min firm number:   {counts.min()}')
print(f'mean firm number:  {int(counts.mean())}')

number of signals: 157
number of firms:   5598
max firm number:   1871
min firm number:   1219
mean firm number:  1421
